## Spatio-Temporal Beam Prediction
**Architecture:** Sliding-window multimodal input → per-frame spatial encoders → cross-modal attention → 1-layer GRU → Dense(64) → beam_index head

**Purpose:** Compare temporal (GRU) vs. frame-by-frame (MultimodalBeamModel) for FL beam prediction.

Key additions over `scaled_beam_only.ipynb`:
- `SlidingWindowMultimodalDataset` — produces `(W, ...)` windows with no trajectory leakage
- `SpatioTemporalBeamModel` — per-step spatial encoding + GRU temporal model
- `SpatioTemporalFlowerClient` — FL client tracking **Beam Flicker Rate**


!pip install "protobuf<6.0.0dev" flwr ray

from google.colab import drive
# Mount Google Drive
drive.mount('/content/drive')

In [1]:
import numpy as np
import random
import os
import re
import io
import zipfile
import time
from collections import defaultdict
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path, PurePosixPath
from typing import Dict, List, Optional, Sequence, Tuple, Union

import flwr as fl
import ray
import tensorflow as tf
from tensorflow import keras
import pandas as pd
import yaml

seed = 42
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['PYTHONHASHSEED'] = str(seed)
tf.keras.backend.clear_session()
print('TF version:', tf.__version__, ' | Flwr version:', fl.__version__)


TF version: 2.19.1  | Flwr version: 1.22.0


In [13]:
CFG = {
    # Training
    'local_epochs'   : 3,
    'lr'             : 5e-4,
    'batch_size'     : 32,
    'grad_clip_norm' : 5.0,
    'label_smoothing': 0.1,

    # Federated
    'client_frac'    : 1.0,

    # Codebook
    'Q_tx'           : 16,
    'Q_rx'           : 16,

    # Temporal window
    'window_size'    : 5,

    # Model — trimmed for FL (~1M params, 2.3× original)
    # Structural improvements kept; excess width removed.
    'gru_units'      : 160,    # single GRU (vs 2-layer 256) — halves GRU cost
    'emb_dim'        : 160,    # 128→160, not 192 — meaningful gain, less overhead
    'dropout'        : 0.2,
}

MAX_CLIENTS = 50

print('CFG loaded.')
print(f'  emb_dim={CFG["emb_dim"]}  gru_units={CFG["gru_units"]}  '
      f'dropout={CFG["dropout"]}  lr={CFG["lr"]}  label_smoothing={CFG["label_smoothing"]}')
print(f'  window_size={CFG["window_size"]}  MAX_CLIENTS={MAX_CLIENTS}')
print(f'  target ~1M params (2.3× original, 0.43× v2)')

CFG loaded.
  emb_dim=160  gru_units=160  dropout=0.2  lr=0.0005  label_smoothing=0.1
  window_size=5  MAX_CLIENTS=50
  target ~1M params (2.3× original, 0.43× v2)


In [14]:
def _parse_total_antennas(config_name: str, side: str) -> int:
    m = re.search(rf'{side}_(\d+)_(\d+)', config_name)
    if not m:
        raise ValueError(f'Could not parse {side} from: {config_name}')
    return int(m.group(1)) * int(m.group(2))


def _safe_isfinite(x):
    if np.iscomplexobj(x):
        return np.isfinite(x.real).all() and np.isfinite(x.imag).all()
    return np.isfinite(x).all()


@dataclass(frozen=True)
class ChannelSampleRef:
    zip_path: Path
    inner_npz: str


def generate_dft_codebook(size: int, num_beams: int) -> np.ndarray:
    n = np.arange(size).reshape(-1, 1)
    q = np.arange(num_beams).reshape(1, -1)
    codebook = (1.0 / np.sqrt(size)) * np.exp(1j * (2 * np.pi / num_beams) * n * q)
    return codebook.astype(np.complex64)


_N_LIDAR_PTS = 512
_IMU_DIM     = 7


class ChannelDataset:
    CHANNEL_PATH_VARIANTS = ['Channel Data/V2I', 'Channel Data']

    def __init__(self, root, config_name, Q_tx=16, Q_rx=16,
                 weather_conditions=None, towns=None,
                 scenario_contains=None, cav_contains=None,
                 stride=1, assert_no_nans=True, assert_shapes=True):
        self.root = Path(root)
        self.config_name = config_name
        self.weather_conditions = list(weather_conditions or ['sunny'])
        self.config_dirs = []
        for weather in self.weather_conditions:
            for variant in self.CHANNEL_PATH_VARIANTS:
                cdir = self.root / weather / variant / config_name
                if cdir.exists():
                    self.config_dirs.append(cdir)
                    print(f'  [OK] {weather} -> {cdir}')
                    break
        if not self.config_dirs:
            raise FileNotFoundError(f'No channel data for: {self.weather_conditions}')
        self.nt = _parse_total_antennas(config_name, 'Nt')
        self.nr = _parse_total_antennas(config_name, 'Nr')
        self.Q_tx, self.Q_rx = Q_tx, Q_rx
        self.towns = list(towns) if towns else None
        self.scenario_contains = scenario_contains
        self.cav_contains = cav_contains
        self.stride = max(1, int(stride))
        self.assert_no_nans = assert_no_nans
        self.assert_shapes = assert_shapes
        self.tx_codebook = generate_dft_codebook(self.nt, Q_tx)
        self.rx_codebook = generate_dft_codebook(self.nr, Q_rx)
        self.index = self._build_index()
        if self.stride > 1:
            self.index = self.index[::self.stride]
        self._expected_csi_shape = None

    def _build_index(self):
        refs = []
        for config_dir in self.config_dirs:
            zips = sorted(config_dir.glob('Town*.zip')) if self.towns is None \
                   else [config_dir / f'{t}.zip' for t in self.towns]
            for zp in zips:
                if not zp.exists(): continue
                with zipfile.ZipFile(zp, 'r') as z:
                    for name in z.namelist():
                        if not name.endswith('_paths.npz'): continue
                        p = PurePosixPath(name)
                        if self.scenario_contains and self.scenario_contains.lower() not in str(p).lower(): continue
                        if self.cav_contains and self.cav_contains.lower() not in str(p).lower(): continue
                        refs.append(ChannelSampleRef(zip_path=zp, inner_npz=name))
        def sort_key(r):
            p = PurePosixPath(r.inner_npz)
            m = re.match(r'(\d+)_paths$', p.stem)
            return (str(r.zip_path), str(p.parent), int(m.group(1)) if m else -1)
        refs.sort(key=sort_key)
        if not refs: raise ValueError(f'No *_paths.npz found under: {self.config_dirs}')
        return refs

    def _parse_metadata(self, inner_path):
        p = PurePosixPath(inner_path)
        m = re.match(r'(\d+)_paths\.npz$', p.name)
        frame_id = int(m.group(1)) if m else -1
        cav_id   = p.parent.name if 'cav' in p.parent.name.lower() else 'unknown'
        location = p.parent.parent.name or 'unknown'
        return {'location': location, 'cav_id': cav_id, 'frame_id': frame_id}

    def get_sample_metadata(self, idx):
        ref  = self.index[idx]
        meta = self._parse_metadata(ref.inner_npz)
        meta['town']       = ref.zip_path.stem
        meta['zip_path']   = str(ref.zip_path)
        meta['inner_path'] = ref.inner_npz
        for weather in self.weather_conditions:
            if weather in str(ref.zip_path):
                meta['weather'] = weather; break
        else:
            meta['weather'] = 'unknown'
        return meta

    def build_metadata_index(self):
        rows = []
        for idx in range(len(self)):
            meta = self.get_sample_metadata(idx)
            meta['sample_idx'] = idx
            rows.append(meta)
        return pd.DataFrame(rows)

    def __len__(self): return len(self.index)

    def _load_npz(self, ref):
        with zipfile.ZipFile(ref.zip_path, 'r') as z:
            raw = z.read(ref.inner_npz)
        npz = np.load(io.BytesIO(raw))
        return {k: npz[k] for k in npz.files}

    def _extract_csi(self, arrays):
        a = arrays['a']
        if self.assert_no_nans: assert _safe_isfinite(a)
        a_sq = np.squeeze(a).astype(np.complex64)
        if a_sq.ndim == 2: a_sq = a_sq[:, :, np.newaxis]
        H   = np.sum(a_sq, axis=2)
        csi = np.stack([H.real, H.imag, np.abs(H)], axis=-1).astype(np.float32)
        if self.assert_no_nans: assert _safe_isfinite(csi)
        return csi


    def compute_beam_index(self, a_sq):
        H = np.sum(a_sq.astype(np.complex64), axis=2)
        response = (self.rx_codebook.conj().T @ H @ self.tx_codebook).astype(np.complex64, copy=False)
        power = np.abs(response) ** 2
        gain_per_tx = power.max(axis=0)
        beam_index = int(gain_per_tx.argmax())
        g_opt = float(power.max())
        assert np.isfinite(g_opt) and g_opt >= 0
        return beam_index, g_opt, H

    def compute_los_label(self, a_sq, tau, kappa=0.5, ds_thresh=50e-9):
        eps = 1e-12
        path_power = np.sum(np.abs(a_sq.astype(np.complex64)) ** 2, axis=(0, 1)).astype(np.float64)

        tau = np.asarray(tau).squeeze().astype(np.float64)
        if tau.ndim == 0:
            tau = np.array([float(tau)], dtype=np.float64)

        n = min(path_power.shape[0], tau.shape[0])
        path_power = path_power[:n]
        tau = tau[:n]

        total_power = float(path_power.sum())
        l0 = int(np.argmin(tau))
        dom_ratio = float(path_power[l0] / (total_power + eps))

        tau_bar = float(np.sum(path_power * tau) / (total_power + eps))
        delay_spread = float(np.sqrt(np.sum(path_power * (tau - tau_bar) ** 2) / (total_power + eps)))

        los = int((dom_ratio > kappa) and (delay_spread < ds_thresh))

        assert np.isfinite(dom_ratio) and 0.0 <= dom_ratio <= 1.0 + 1e-6
        assert np.isfinite(delay_spread) and delay_spread >= 0.0
        return los, dom_ratio, delay_spread

    def __getitem__(self, idx):
        ref    = self.index[idx]
        arrays = self._load_npz(ref)
        a_sq   = np.squeeze(arrays['a']).astype(np.complex64)
        if a_sq.ndim == 2: a_sq = a_sq[:, :, np.newaxis]
        csi_tensor    = self._extract_csi(arrays)
        if self.assert_shapes:
            if self._expected_csi_shape is None:
                self._expected_csi_shape = tuple(csi_tensor.shape)
            else:
                assert tuple(csi_tensor.shape) == self._expected_csi_shape

        beam_idx, g_opt, H = self.compute_beam_index(a_sq)
        los, dom_ratio, delay_spread = self.compute_los_label(a_sq, arrays['tau'])

        labels = {
            'beam_index': np.array(beam_idx, dtype=np.int64),
            'g_opt': np.array(g_opt, dtype=np.float32),
            'los': np.array(los, dtype=np.int64),
            'dom_ratio': np.array(dom_ratio, dtype=np.float32),
            'delay_spread': np.array(delay_spread, dtype=np.float32),
            'H_complex': H.astype(np.complex64, copy=False),
        }
        return csi_tensor, labels


print('ChannelDataset, generate_dft_codebook defined.')


ChannelDataset, generate_dft_codebook defined.


In [15]:
SENSOR_PATH_VARIANTS = ['Sensor Data']


@dataclass
class SensorZipEntry:
    zip_path: Path
    scenario_prefix: str
    channel_location: str
    town: str


class SensorIndex:
    def __init__(self, root, weather_conditions, towns):
        self.root = Path(root)
        self._map: Dict[Tuple, SensorZipEntry] = {}
        self._build(weather_conditions, towns)

    def _build(self, weathers, towns):
        for weather in weathers:
            for variant in SENSOR_PATH_VARIANTS:
                for town in towns:
                    sensor_dir = self.root / weather / variant / town
                    if not sensor_dir.exists(): continue
                    for zp in sorted(sensor_dir.glob('*.zip')):
                        stem   = zp.stem
                        ch_loc = self._stem_to_channel_location(stem, town)
                        entry  = SensorZipEntry(zp, stem, ch_loc, town)
                        self._map[(weather, ch_loc, town)] = entry
        print(f'[SensorIndex] {len(self._map)} sensor zips indexed')

    @staticmethod
    def _stem_to_channel_location(stem, town):
        rest  = stem[len(town) + 1:]
        parts = rest.split('_')
        loc_parts, stop = [], {'wiz', 'seed', 'slope'}
        for p in parts:
            if p.lower() in stop or re.match(r'seed\d+', p.lower()): break
            loc_parts.append(p)
        return f'{town}_{"_".join(loc_parts)}'

    def resolve(self, weather, channel_inner_npz):
        p = PurePosixPath(channel_inner_npz)
        if len(p.parts) < 4: return None
        town, ch_location, cav_id = p.parts[0], p.parts[1], p.parts[2]
        m = re.match(r'(\d+)_paths\.npz$', p.parts[3])
        if not m: return None
        entry = self._map.get((weather, ch_location, town))
        return (entry, cav_id, m.group(1)) if entry else None


def _read_pcd_xyz(raw_bytes):
    lines = raw_bytes.split(b'\n')
    header_end, n_points = 0, 0
    for i, line in enumerate(lines):
        text = line.decode(errors='replace').strip()
        if text.startswith('POINTS'): n_points = int(text.split()[1])
        if text == 'DATA ascii': header_end = i + 1; break
    pts = []
    for line in lines[header_end: header_end + n_points]:
        parts = line.split()
        if len(parts) >= 3: pts.append([float(parts[0]), float(parts[1]), float(parts[2])])
    return np.array(pts, dtype=np.float32) if pts else np.zeros((0, 3), np.float32)


def _fast_parse_imu(raw_bytes):
    text = raw_bytes.decode('utf-8', errors='replace')
    if 'data_missing: true' in text or 'data_missing: True' in text:
        return np.zeros(_IMU_DIM, np.float32)
    vals, section = {}, None
    for line in text.split('\n'):
        s = line.strip()
        if s.startswith('accelerometer:'): section = 'acc'
        elif s.startswith('gyroscope:'): section = 'gyro'
        elif s.startswith('compass:'):
            try: vals['compass'] = float(s.split(':', 1)[1].strip())
            except ValueError: vals['compass'] = 0.0
        elif section and ':' in s:
            key, val = s.split(':', 1)
            key = key.strip()
            if key in ('x', 'y', 'z'):
                try: vals[f'{section}_{key}'] = float(val.strip())
                except ValueError: pass
    return np.array([
        vals.get('acc_x', 0) / 9.81, vals.get('acc_y', 0) / 9.81, vals.get('acc_z', 0) / 9.81,
        vals.get('gyro_x', 0), vals.get('gyro_y', 0), vals.get('gyro_z', 0),
        vals.get('compass', 0) / 360.0,
    ], dtype=np.float32)


class MultimodalChannelDataset:
    def __init__(self, channel_dataset, sensor_index, n_lidar_pts=_N_LIDAR_PTS, seed=42):
        self.channel_ds  = channel_dataset
        self.sensor_idx  = sensor_index
        self.n_lidar_pts = n_lidar_pts
        self._rng        = np.random.default_rng(seed)
        self._sensor_map = self._build_sensor_map()
        matched = sum(1 for v in self._sensor_map if v is not None)
        print(f'[MultimodalDataset] {len(self)} samples | sensor-aligned: {matched} | zeros: {len(self)-matched}')

    def _build_sensor_map(self):
        mapping = []
        for idx in range(len(self.channel_ds)):
            ref  = self.channel_ds.index[idx]
            meta = self.channel_ds.get_sample_metadata(idx)
            mapping.append(self.sensor_idx.resolve(meta.get('weather', 'sunny'), ref.inner_npz))
        return mapping

    def __len__(self): return len(self.channel_ds)

    def __getitem__(self, idx):
        csi, labels = self.channel_ds[idx]
        resolved = self._sensor_map[idx]
        if resolved is not None:
            entry, cav_id, frame_id = resolved
            try:
                with zipfile.ZipFile(entry.zip_path, 'r') as z:
                    prefix = f'{entry.scenario_prefix}/{cav_id}/{frame_id}'
                    pts = _read_pcd_xyz(z.read(f'{prefix}.pcd'))
                    if len(pts) == 0: pts = np.zeros((1, 3), np.float32)
                    if len(pts) >= self.n_lidar_pts:
                        idx_pts = self._rng.choice(len(pts), self.n_lidar_pts, replace=False)
                        pts = pts[idx_pts]
                    else:
                        pts = np.concatenate([pts, np.zeros((self.n_lidar_pts - len(pts), 3), np.float32)])
                    mean = pts.mean(0, keepdims=True); std = pts.std(0, keepdims=True) + 1e-8
                    lidar = ((pts - mean) / std).astype(np.float32)
                    imu = _fast_parse_imu(z.read(f'{prefix}.yaml'))
            except Exception:
                lidar = np.zeros((self.n_lidar_pts, 3), np.float32)
                imu   = np.zeros(_IMU_DIM, np.float32)
        else:
            lidar = np.zeros((self.n_lidar_pts, 3), np.float32)
            imu   = np.zeros(_IMU_DIM, np.float32)
        return csi, lidar, imu, labels

    @property
    def tx_codebook(self): return self.channel_ds.tx_codebook
    @property
    def rx_codebook(self): return self.channel_ds.rx_codebook
    @property
    def nr(self): return self.channel_ds.nr
    @property
    def nt(self): return self.channel_ds.nt


print('SensorIndex, MultimodalChannelDataset defined.')


SensorIndex, MultimodalChannelDataset defined.


In [16]:
class DatasetSplitter:
    def __init__(self, dataset: ChannelDataset):
        self.dataset     = dataset
        self.metadata_df = dataset.build_metadata_index()
        print(f'Metadata index: {len(self.metadata_df)} samples')

    def get_trajectory_groups(self):
        trajectories = {}
        group_cols = ['weather', 'town', 'location', 'cav_id'] \
            if 'weather' in self.metadata_df.columns else ['town', 'location', 'cav_id']
        for keys, group in self.metadata_df.groupby(group_cols):
            traj_id = '_'.join(str(k) for k in keys) if isinstance(keys, tuple) else str(keys)
            trajectories[traj_id] = group.sort_values('frame_id')['sample_idx'].tolist()
        return trajectories

    def split_trajectory_temporal(self, indices, train_ratio=0.7):
        split = int(len(indices) * train_ratio)
        return indices[:split], indices[split:]

    def split_trajectory_random(self, indices, train_ratio=0.7, seed=42):
        rng = np.random.RandomState(seed)
        shuffled = np.array(indices.copy()); rng.shuffle(shuffled)
        split = int(len(shuffled) * train_ratio)
        return shuffled[:split].tolist(), shuffled[split:].tolist()


@dataclass
class ChannelClientData:
    train_indices: List[int]
    test_indices:  List[int]
    client_id:     int
    trajectory_id: str


def build_clients(dataset, train_ratio=0.7, min_trajectory_length=10,
                  split_strategy='random'):
    splitter    = DatasetSplitter(dataset)
    trajectories = {t: idx for t, idx in splitter.get_trajectory_groups().items()
                    if len(idx) >= min_trajectory_length}
    print(f'Trajectories after filter: {len(trajectories)}')
    clients = []
    for cid, traj_id in enumerate(sorted(trajectories)):
        if split_strategy == 'random':
            train_idx, test_idx = splitter.split_trajectory_random(trajectories[traj_id], train_ratio)
        else:
            train_idx, test_idx = splitter.split_trajectory_temporal(trajectories[traj_id], train_ratio)
        if not train_idx or not test_idx: continue
        clients.append(ChannelClientData(train_idx, test_idx, cid, traj_id))
    print(f'Clients: {len(clients)}')
    return clients


print('DatasetSplitter, build_clients defined.')


DatasetSplitter, build_clients defined.


In [17]:
# ── Sliding Window Dataset ───────────────────────────────────────────────────
# A window = W consecutive frames from the SAME trajectory.
# Target = beam_index of the LAST frame in the window.
# Splitting: trajectory-level temporal split BEFORE windowing prevents leakage.


class SlidingWindowMultimodalDataset:
    """
    Builds sliding windows over preloaded multimodal data.

    Parameters
    ----------
    preloaded_cache : dict {global_idx: (csi, lidar, imu, labels)}
    indices         : list of global sample indices for this split
    metadata_df     : full ChannelDataset metadata DataFrame
    window_size     : W — frames per window

    __getitem__ returns:
        csi_seq   : (W, Nr, Nt, 3)   float32
        lidar_seq : (W, N_pts, 3)    float32
        imu_seq   : (W, 7)           float32
        beam_label: int  (beam at last frame)
    """

    def __init__(self, preloaded_cache, indices, metadata_df, window_size=5):
        self.cache       = preloaded_cache
        self.window_size = window_size
        self.windows: List[List[int]] = []   # each element: [idx_t0, ..., idx_tW-1]

        idx_set   = set(indices)
        local_meta = metadata_df[metadata_df['sample_idx'].isin(idx_set)].copy()

        group_cols = ['weather', 'town', 'location', 'cav_id'] \
            if 'weather' in local_meta.columns else ['town', 'location', 'cav_id']

        for _, group in local_meta.groupby(group_cols):
            traj_indices = group.sort_values('frame_id')['sample_idx'].tolist()
            for start in range(len(traj_indices) - window_size + 1):
                self.windows.append(traj_indices[start:start + window_size])

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        window = self.windows[idx]
        csi_seq, lidar_seq, imu_seq = [], [], []
        for frame_idx in window:
            csi, lidar, imu, _ = self.cache[frame_idx]
            csi_seq.append(csi)
            lidar_seq.append(lidar)
            imu_seq.append(imu)
        beam_label = int(self.cache[window[-1]][3]['beam_index'])
        return (
            np.stack(csi_seq),    # (W, Nr, Nt, 3)
            np.stack(lidar_seq),  # (W, N_pts, 3)
            np.stack(imu_seq),    # (W, 7)
            beam_label,
        )


@dataclass
class WindowedClientData:
    train_windows: List[List[int]]  # windows for training
    test_windows:  List[List[int]]  # windows for eval (sorted by time)
    tx_codebook:   np.ndarray
    rx_codebook:   np.ndarray
    client_id:     int
    trajectory_id: str


def build_windowed_clients(
    preloaded_cache: dict,
    channel_dataset: ChannelDataset,
    window_size: int = 5,
    train_ratio: float = 0.7,
    min_trajectory_length: int = 20,
) -> List[WindowedClientData]:
    """
    Build federated clients with sliding window splits.

    Temporal split → train windows and test windows never overlap.
    Minimum trajectory length ensures enough frames for at least one
    window in both train and test splits.
    """
    splitter      = DatasetSplitter(channel_dataset)
    traj_groups   = splitter.get_trajectory_groups()
    tx_cb         = channel_dataset.tx_codebook
    rx_cb         = channel_dataset.rx_codebook

    clients, skipped = [], 0
    min_len = max(min_trajectory_length, window_size * 2 + 2)

    for traj_id in sorted(traj_groups):
        indices = [i for i in traj_groups[traj_id] if i in preloaded_cache]
        if len(indices) < min_len:
            skipped += 1
            continue

        # Temporal split (must come BEFORE windowing to prevent leakage)
        train_idx, test_idx = splitter.split_trajectory_temporal(indices, train_ratio)

        if len(train_idx) < window_size or len(test_idx) < window_size:
            skipped += 1
            continue

        # Build sliding windows within each split
        train_windows = [train_idx[s:s + window_size]
                         for s in range(len(train_idx) - window_size + 1)]
        test_windows  = [test_idx[s:s + window_size]
                         for s in range(len(test_idx)  - window_size + 1)]

        if not train_windows or not test_windows:
            skipped += 1
            continue

        clients.append(WindowedClientData(
            train_windows = train_windows,
            test_windows  = test_windows,
            tx_codebook   = tx_cb,
            rx_codebook   = rx_cb,
            client_id     = len(clients),
            trajectory_id = traj_id,
        ))

    total_train = sum(len(c.train_windows) for c in clients)
    total_test  = sum(len(c.test_windows)  for c in clients)
    print(f'Windowed clients: {len(clients)}  (skipped: {skipped})')
    print(f'Total train windows: {total_train}  |  test windows: {total_test}')
    print(f'Avg windows/client:  {(total_train+total_test)/max(len(clients),1):.0f}')
    return clients


print('SlidingWindowMultimodalDataset, WindowedClientData, build_windowed_clients defined.')


# ── Shared window-loading helper ─────────────────────────────────────────────
# Defined here (before both FL clients) so both can import it.



def _load_windows_from_cache(
    cache: dict,
    windows: List[List[int]],
    normalize_csi: bool = True,
    return_H: bool = False,
):
    """
    Stack a list of index-windows from the preloaded cache into batch arrays.

    Parameters
    ----------
    cache   : {global_idx: (csi, lidar, imu, labels)}
    windows : list of [idx_t0, ..., idx_tW-1]

    Returns:
        X = (csi, lidar, imu)
            csi   : (N, W, Nr, Nt, 3)  float32
            lidar : (N, W, N_pts, 3)   float32
            imu   : (N, W, 7)          float32
        y = {
            "beam_index": (N,) int32,
            "g_opt": (N,) float32,
            "los": (N,) int32,
            "beam_change": (N,) int32,
        }
        Optional H: (N, Nr, Nt) complex64 if return_H=True
    """
    CSI, LIDAR, IMU = [], [], []
    LABELS_BEAM, G_OPTS, LOSS, BEAM_CHANGES, HS = [], [], [], [], []

    for window in windows:
        csi_seq, lidar_seq, imu_seq = [], [], []
        for frame_idx in window:
            csi, lidar, imu, _ = cache[frame_idx]
            csi_seq.append(csi)
            lidar_seq.append(lidar)
            imu_seq.append(imu)

        CSI.append(np.stack(csi_seq))
        LIDAR.append(np.stack(lidar_seq))
        IMU.append(np.stack(imu_seq))

        last_labels = cache[window[-1]][3]
        last_beam = int(last_labels['beam_index'])
        prev_beam = int(cache[window[-2]][3]['beam_index']) if len(window) >= 2 else last_beam
        beam_change = int(last_beam != prev_beam)

        LABELS_BEAM.append(last_beam)
        G_OPTS.append(float(last_labels['g_opt']))
        LOSS.append(int(last_labels['los']))
        BEAM_CHANGES.append(beam_change)

        if return_H:
            HS.append(last_labels['H_complex'])

    csi_arr   = np.stack(CSI).astype(np.float32)
    lidar_arr = np.stack(LIDAR).astype(np.float32)
    imu_arr   = np.stack(IMU).astype(np.float32)

    if normalize_csi:
        N, W = csi_arr.shape[:2]
        flat = csi_arr.reshape(N * W, *csi_arr.shape[2:])
        flat = (flat - flat.mean(axis=(1, 2), keepdims=True)) / (
            flat.std(axis=(1, 2), keepdims=True) + 1e-8
        )
        csi_arr = flat.reshape(N, W, *csi_arr.shape[2:])

    y = {
        'beam_index': np.array(LABELS_BEAM, dtype=np.int32),
        'g_opt': np.array(G_OPTS, dtype=np.float32),
        'los': np.array(LOSS, dtype=np.int32),
        'beam_change': np.array(BEAM_CHANGES, dtype=np.int32),
    }

    X = (csi_arr, lidar_arr, imu_arr)
    if return_H:
        return X, y, np.stack(HS).astype(np.complex64)
    return X, y


print('_load_windows_from_cache defined.')


SlidingWindowMultimodalDataset, WindowedClientData, build_windowed_clients defined.
_load_windows_from_cache defined.


In [18]:
# ── Spatial Encoders + Cross-Modal Fusion ────────────────────────────────────
#
# Design goal: keep every structural improvement from the upgrade, trim width.
#
#  ChannelEncoder  — 4-layer Conv2D + concat(GAP,GMP) → Dense(512→D)
#                    The concat pool is the single biggest accuracy gain;
#                    the 4th conv adds receptive field on the 16×16 grid.
#                    Removed the intermediate Dense(512→512) from v2 — wasteful.
#
#  CrossModalFusion — Transformer block: MHA(4 heads) + FFN(2×) + residuals
#                     Reduced from 8 heads / FFN(4×) in v2 — those were overkill
#                     for 3 tokens. 4 heads / FFN(2×) is the standard for small
#                     token counts and still far better than plain mean-pooling.


@tf.keras.utils.register_keras_serializable()
class ChannelEncoder(tf.keras.Model):
    """CNN encoder for CSI. Input: (B, Nr, Nt, 3)  Output: (B, emb_dim)"""
    def __init__(self, nr, nt, emb_dim=160, dropout=0.0, **kwargs):
        super().__init__(**kwargs)
        self.conv1 = keras.layers.Conv2D(32,  3, padding='same', activation='relu')
        self.bn1   = keras.layers.BatchNormalization()
        self.conv2 = keras.layers.Conv2D(64,  3, padding='same', activation='relu')
        self.bn2   = keras.layers.BatchNormalization()
        self.conv3 = keras.layers.Conv2D(128, 3, padding='same', activation='relu')
        self.bn3   = keras.layers.BatchNormalization()
        self.conv4 = keras.layers.Conv2D(256, 3, padding='same', activation='relu')
        self.bn4   = keras.layers.BatchNormalization()
        self.gap   = keras.layers.GlobalAveragePooling2D()
        self.gmp   = keras.layers.GlobalMaxPooling2D()   # captures peak beam energy
        # GAP+GMP concat → 512; project directly to emb_dim (no 512→512 bottleneck)
        self.proj  = keras.Sequential([
            keras.layers.Dense(512, activation='relu'),
            keras.layers.Dropout(dropout),
            keras.layers.Dense(emb_dim),
        ])

    def call(self, x, training=False):
        h = self.bn1(self.conv1(x), training=training)
        h = self.bn2(self.conv2(h), training=training)
        h = self.bn3(self.conv3(h), training=training)
        h = self.bn4(self.conv4(h), training=training)
        pooled = tf.concat([self.gap(h), self.gmp(h)], axis=-1)  # (B, 512)
        return self.proj(pooled, training=training)


@tf.keras.utils.register_keras_serializable()
class LiDAREncoder(tf.keras.Model):
    """PointNet-style encoder. Input: (B, N, 3). Output: (B, emb_dim)."""
    def __init__(self, emb_dim=160, dropout=0.0, **kwargs):
        super().__init__(**kwargs)
        self.conv1 = keras.layers.Conv1D(64,  1, activation='relu')
        self.bn1   = keras.layers.BatchNormalization()
        self.conv2 = keras.layers.Conv1D(128, 1, activation='relu')
        self.bn2   = keras.layers.BatchNormalization()
        self.conv3 = keras.layers.Conv1D(256, 1, activation='relu')
        self.bn3   = keras.layers.BatchNormalization()
        self.pool  = keras.layers.GlobalMaxPooling1D()
        self.proj  = keras.Sequential([
            keras.layers.Dense(256, activation='relu'),
            keras.layers.Dropout(dropout),
            keras.layers.Dense(emb_dim),
        ])

    def call(self, x, training=False):
        h = self.bn1(self.conv1(x), training=training)
        h = self.bn2(self.conv2(h), training=training)
        h = self.bn3(self.conv3(h), training=training)
        return self.proj(self.pool(h), training=training)


@tf.keras.utils.register_keras_serializable()
class IMUEncoder(tf.keras.Model):
    """MLP encoder for IMU (7-dim). Output: (B, emb_dim)."""
    def __init__(self, emb_dim=160, dropout=0.0, **kwargs):
        super().__init__(**kwargs)
        self.net = keras.Sequential([
            keras.layers.Dense(64,  activation='relu'),
            keras.layers.Dropout(dropout),
            keras.layers.Dense(128, activation='relu'),
            keras.layers.Dense(emb_dim),
        ])

    def call(self, x, training=False):
        return self.net(x, training=training)


@tf.keras.utils.register_keras_serializable()
class CrossModalFusion(tf.keras.layers.Layer):
    """
    Lightweight Transformer block over 3 modality tokens.

    MHA(4 heads) + FFN(2×width) + residuals + weighted pool.
    4 heads / 2× FFN chosen deliberately: for M=3 tokens, wider heads and
    deeper FFN add parameters without improving expressiveness.

    Input:  (B, 3, emb_dim)   Output: (B, emb_dim)
    """
    def __init__(self, emb_dim=160, n_heads=4, dropout=0.0, **kwargs):
        super().__init__(**kwargs)
        self.mha   = keras.layers.MultiHeadAttention(
            num_heads=n_heads, key_dim=emb_dim // n_heads, dropout=dropout)
        self.norm1 = keras.layers.LayerNormalization()
        self.ffn   = keras.Sequential([
            keras.layers.Dense(emb_dim * 2, activation='gelu'),
            keras.layers.Dropout(dropout),
            keras.layers.Dense(emb_dim),
        ])
        self.norm2   = keras.layers.LayerNormalization()
        self.token_w = keras.layers.Dense(1)   # learnable per-token importance

    def call(self, x, training=False):
        # x: (B, 3, D)
        x = self.norm1(x + self.mha(x, x, training=training))
        x = self.norm2(x + self.ffn(x, training=training))
        w = tf.nn.softmax(self.token_w(x), axis=1)   # (B, 3, 1)
        return tf.reduce_sum(x * w, axis=1)           # (B, D)


print('ChannelEncoder  — 4×Conv2D + GAP+GMP + Dense(512→D)')
print('LiDAREncoder    — 3×Conv1D + GMP + Dense(emb_dim)')
print('IMUEncoder      — MLP 64→128→emb_dim')
print('CrossModalFusion— Transformer: MHA(4h) + FFN(2×) + weighted pool')

ChannelEncoder  — 4×Conv2D + GAP+GMP + Dense(512→D)
LiDAREncoder    — 3×Conv1D + GMP + Dense(emb_dim)
IMUEncoder      — MLP 64→128→emb_dim
CrossModalFusion— Transformer: MHA(4h) + FFN(2×) + weighted pool


In [19]:
# ── Spatio-Temporal Beam Model (trimmed) ─────────────────────────────────────
#
# Architecture:
#   Per time-step (reshape trick):
#     ChannelEncoder(4 conv + GAP+GMP) → (B*W, D)
#     LiDAREncoder(PointNet)           → (B*W, D)
#     IMUEncoder(MLP)                  → (B*W, D)
#     stack → (B*W, 3, D) → CrossModalFusion → (B*W, D)
#
#   Reshape → (B, W, D); LayerNorm per timestep
#
#   Single GRU(D)  — single wider GRU vs 2-layer 256
#                    (738k → 293k for GRU alone, biggest saving)
#
#   Trunk: Dense(256, relu) → BN → Dropout → Dense(128, relu) → Dense(n_beams)
#
# Parameter budget comparison:
#   v1 original  ~452k   (collapsed)
#   v2 upgraded  ~2.4M   (too heavy for FL)
#   this version ~1.0M   (2.3× original, 0.43× v2)


@tf.keras.utils.register_keras_serializable()
class SpatioTemporalBeamModel(keras.Model):
    """
    Spatio-temporal beam prediction (trimmed).

    call(csi_seq, lidar_seq, imu_seq, training)
        csi_seq   : (B, W, Nr, Nt, 3)
        lidar_seq : (B, W, N_pts, 3)
        imu_seq   : (B, W, 7)
        -> logits : (B, n_beams)
    """

    def __init__(self, nr=16, nt=16, n_beams=16, window_size=5,
                 emb_dim=160, gru_units=160, dropout=0.2, **kwargs):
        super().__init__(**kwargs)
        self.nr          = nr
        self.nt          = nt
        self.n_beams     = n_beams
        self.window_size = window_size
        self.emb_dim     = emb_dim
        self.gru_units   = gru_units

        # Spatial encoders
        self.csi_enc   = ChannelEncoder(nr, nt, emb_dim=emb_dim, dropout=dropout)
        self.lidar_enc = LiDAREncoder(emb_dim=emb_dim, dropout=dropout)
        self.imu_enc   = IMUEncoder(emb_dim=emb_dim, dropout=dropout)

        # Cross-modal Transformer fusion (lightweight: 4 heads, FFN×2)
        self.fusion   = CrossModalFusion(emb_dim=emb_dim, n_heads=4, dropout=dropout)

        # Normalise per-timestep embeddings before GRU
        self.seq_norm = keras.layers.LayerNormalization()

        # Single GRU (wider is better value than stacked for this sequence length)
        self.gru  = keras.layers.GRU(gru_units, return_sequences=False,
                                     dropout=dropout, recurrent_dropout=0.0)

        # Prediction trunk — same capacity as scaled_beam_only's working head
        self.trunk = keras.Sequential([
            keras.layers.Dense(256, activation='relu'),
            keras.layers.BatchNormalization(),
            keras.layers.Dropout(dropout),
            keras.layers.Dense(128, activation='relu'),
            keras.layers.Dropout(dropout),
            keras.layers.Dense(n_beams, name='beam_head'),
        ])

    def call(self, csi_seq, lidar_seq, imu_seq, training=False):
        B = tf.shape(csi_seq)[0]
        W = self.window_size
        D = self.emb_dim

        # ── Spatial encoding (reshape folds W into batch dim) ────────────
        csi_flat   = tf.reshape(csi_seq,   [B * W, self.nr, self.nt, 3])
        lidar_flat = tf.reshape(lidar_seq, [B * W, _N_LIDAR_PTS, 3])
        imu_flat   = tf.reshape(imu_seq,   [B * W, _IMU_DIM])

        e_csi   = self.csi_enc(csi_flat,    training=training)   # (B*W, D)
        e_lidar = self.lidar_enc(lidar_flat, training=training)   # (B*W, D)
        e_imu   = self.imu_enc(imu_flat,    training=training)   # (B*W, D)

        # ── Cross-modal Transformer fusion per time step ──────────────────
        e_stack  = tf.stack([e_csi, e_lidar, e_imu], axis=1)    # (B*W, 3, D)
        e_fused  = self.fusion(e_stack, training=training)        # (B*W, D)

        # ── Sequence + temporal model ─────────────────────────────────────
        e_seq = self.seq_norm(tf.reshape(e_fused, [B, W, D]))    # (B, W, D)
        h     = self.gru(e_seq, training=training)               # (B, gru_units)

        return self.trunk(h, training=training)                   # (B, n_beams)

    def build_model(self, nr, nt, window_size):
        dummy_csi   = tf.zeros((1, window_size, nr, nt, 3))
        dummy_lidar = tf.zeros((1, window_size, _N_LIDAR_PTS, 3))
        dummy_imu   = tf.zeros((1, window_size, _IMU_DIM))
        _ = self(dummy_csi, dummy_lidar, dummy_imu, training=False)
        n_params = sum(np.prod(w.shape) for w in self.trainable_weights)
        print(f'SpatioTemporalBeamModel built — {n_params:,} parameters')
        self.built = True


_demo = SpatioTemporalBeamModel(
    nr=16, nt=16, n_beams=CFG['Q_tx'], window_size=CFG['window_size'],
    emb_dim=CFG['emb_dim'], gru_units=CFG['gru_units'], dropout=CFG['dropout'],
)
_demo.build_model(16, 16, CFG['window_size'])
print()
print(f'  CSI enc    : 4×Conv2D(32→256) + GAP+GMP + Dense(512→{CFG["emb_dim"]})')
print(f'  LiDAR enc  : 3×Conv1D(64→256) + GMP + Dense({CFG["emb_dim"]})')
print(f'  Fusion     : MHA(4h) + FFN(2×{CFG["emb_dim"]}) + weighted pool')
print(f'  GRU        : single GRU({CFG["gru_units"]})')
print(f'  Trunk      : Dense(256→128→{CFG["Q_tx"]})')

SpatioTemporalBeamModel built — 1,351,281 parameters

  CSI enc    : 4×Conv2D(32→256) + GAP+GMP + Dense(512→160)
  LiDAR enc  : 3×Conv1D(64→256) + GMP + Dense(160)
  Fusion     : MHA(4h) + FFN(2×160) + weighted pool
  GRU        : single GRU(160)
  Trunk      : Dense(256→128→16)


In [20]:
# ── SpatioTemporalFlowerClient ────────────────────────────────────────────────
# Arrays are pre-stacked ONCE before the simulation (see FL training cell).
# No data preparation happens inside fit() or evaluate().


class SpatioTemporalFlowerClient(fl.client.NumPyClient):

    def __init__(self, model, train_data, test_data,
                 tx_codebook, rx_codebook, cfg, trajectory_id):
        """
        Parameters
        ----------
        train_data : ((csi, lidar, imu), y)           — pre-stacked numpy arrays
        test_data  : ((csi, lidar, imu), y, H)        — pre-stacked numpy arrays
        """
        self.model          = model
        self.train_data     = train_data
        self.test_data      = test_data
        self.tx_codebook    = tx_codebook
        self.rx_codebook    = rx_codebook
        self.cfg            = cfg
        self.trajectory_id  = trajectory_id
        self.optimizer      = keras.optimizers.Adam(cfg['lr'])
        self.grad_clip_norm = cfg.get('grad_clip_norm', 5.0)
        self.batch_size     = cfg.get('batch_size', 32)
        self.label_smoothing = cfg.get('label_smoothing', 0.0)
        self.n_beams        = cfg.get('Q_tx', 16)

    def _ensure_built(self):
        if getattr(self.model, 'built', False): return
        self.model.build_model(self.model.nr, self.model.nt, self.model.window_size)

    def _smooth_loss(self, logits, labels):
        """
        Cross-entropy with optional label smoothing.
        Smoothing replaces hard one-hot targets with (1-ε)*one_hot + ε/K,
        which prevents the model from becoming overconfident early → reduces collapse.
        """
        eps = self.label_smoothing
        if eps == 0.0:
            return tf.reduce_mean(
                tf.nn.sparse_softmax_cross_entropy_with_logits(
                    labels=labels, logits=logits))
        # Soft targets: (1-ε)*one_hot + ε/K
        K       = tf.cast(tf.shape(logits)[-1], tf.float32)
        one_hot = tf.one_hot(labels, tf.shape(logits)[-1])
        soft    = (1.0 - eps) * one_hot + eps / K
        log_p   = tf.nn.log_softmax(logits)
        return -tf.reduce_mean(tf.reduce_sum(soft * log_p, axis=-1))

    def compute_normalized_gain(self, H_batch, pred_indices):
        gains = []
        for i in range(len(pred_indices)):
            resp = self.rx_codebook.conj().T @ H_batch[i] @ self.tx_codebook
            g    = np.abs(resp) ** 2
            gains.append(g[:, pred_indices[i]].max() / (g.max() + 1e-12))
        return float(np.mean(gains))

    def get_parameters(self, config):
        self._ensure_built()
        return self.model.get_weights()

    def fit(self, parameters, config):
        self._ensure_built()
        self.model.set_weights(parameters)

        (csi_arr, lidar_arr, imu_arr), y_train = self.train_data
        n, bs = len(csi_arr), self.batch_size
        ep_losses = []

        for _ in range(self.cfg['local_epochs']):
            perm = np.random.permutation(n)
            batch_losses = []
            for start in range(0, n, bs):
                end     = min(start + bs, n)
                idx     = perm[start:end]
                csi_b   = tf.constant(csi_arr[idx],   tf.float32)
                lidar_b = tf.constant(lidar_arr[idx], tf.float32)
                imu_b   = tf.constant(imu_arr[idx],   tf.float32)
                y_b     = tf.constant(y_train['beam_index'][idx], tf.int32)

                with tf.GradientTape() as tape:
                    logits = self.model(csi_b, lidar_b, imu_b, training=True)
                    loss   = self._smooth_loss(logits, y_b)

                grads   = tape.gradient(loss, self.model.trainable_weights)
                safe    = [g if g is not None else tf.zeros_like(v)
                           for g, v in zip(grads, self.model.trainable_weights)]
                clipped, _ = tf.clip_by_global_norm(safe, self.grad_clip_norm)
                self.optimizer.apply_gradients(
                    zip(clipped, self.model.trainable_weights))
                batch_losses.append(float(loss.numpy()))
            ep_losses.append(np.mean(batch_losses))

        return self.model.get_weights(), len(csi_arr), {
            'loss': ep_losses[-1], 'trajectory_id': self.trajectory_id,
        }

    def evaluate(self, parameters, config):
        self._ensure_built()
        self.model.set_weights(parameters)

        (csi_arr, lidar_arr, imu_arr), y_test, H_test = self.test_data
        bs, all_logits = self.batch_size, []

        for start in range(0, len(csi_arr), bs):
            end = min(start + bs, len(csi_arr))
            logits = self.model(
                tf.constant(csi_arr[start:end],   tf.float32),
                tf.constant(lidar_arr[start:end], tf.float32),
                tf.constant(imu_arr[start:end],   tf.float32),
                training=False,
            )
            all_logits.append(logits)

        logits   = tf.concat(all_logits, axis=0)
        y_tf     = tf.constant(y_test['beam_index'], dtype=tf.int32)
        # Evaluate with hard CE (no smoothing) for interpretable metrics
        loss     = tf.reduce_mean(
            tf.nn.sparse_softmax_cross_entropy_with_logits(labels=y_tf, logits=logits))
        preds    = tf.argmax(logits, 1).numpy()
        acc      = float(np.mean(preds == y_test['beam_index']))
        top3     = tf.math.in_top_k(targets=y_tf, predictions=logits, k=3)
        top3_acc = float(tf.reduce_mean(tf.cast(top3, tf.float32)).numpy())
        eval_n   = min(len(preds), 200)
        norm_gain = self.compute_normalized_gain(H_test[:eval_n], preds[:eval_n])
        flicker  = float(np.mean(preds[1:] != preds[:-1])) if len(preds) > 1 else 0.0

        return float(loss.numpy()), len(csi_arr), {
            'beam_accuracy'        : acc,
            'beam_top3_accuracy'   : top3_acc,
            'normalized_gain'      : norm_gain,
            'beam_flicker_rate'    : flicker,
            'beam_num_unique_preds': int(len(np.unique(preds))),
            'loss'                 : float(loss.numpy()),
        }


print('SpatioTemporalFlowerClient defined.')
print(f'  label_smoothing={CFG["label_smoothing"]}  (set via CFG)')

SpatioTemporalFlowerClient defined.
  label_smoothing=0.1  (set via CFG)


In [ ]:
# ── Dataset Loading ──────────────────────────────────────────────────────────

tf.keras.backend.clear_session()

WEATHER_CONDITIONS = ['sunny', 'foggy', 'rainy']
TOWNS = ['Town03', 'Town05', 'Town07', 'Town10']
DATA_ROOT = '/Users/sadmanrahin/Library/CloudStorage/GoogleDrive-saddyrahin2004@gmail.com/My Drive/Dataset'
#DATA_ROOT = '/content/drive/MyDrive/Dataset/'

# 1. Channel dataset
ds = ChannelDataset(
    root               = DATA_ROOT,
    config_name        = 'Nt_1_16_Nr_1_16_fc_28GHz',
    weather_conditions = WEATHER_CONDITIONS,
    towns              = TOWNS,
    stride             = 10,
    Q_tx               = CFG['Q_tx'],
    Q_rx               = CFG['Q_rx'],
)
print(f'Channel dataset: {len(ds)} samples')

# 2. Sensor index
sensor_idx = SensorIndex(
    root               = DATA_ROOT,
    weather_conditions = WEATHER_CONDITIONS,
    towns              = TOWNS,
)

# 3. Multimodal dataset wrapper
mm_ds = MultimodalChannelDataset(channel_dataset=ds, sensor_index=sensor_idx)

# ── Determine which sample indices to preload ─────────────────────────────────
# When MAX_CLIENTS is set, only the first N trajectories' frames are loaded,
# which makes this step much faster for quick experiments.
if MAX_CLIENTS is not None:
    _splitter_pre    = DatasetSplitter(ds)
    _traj_groups_pre = _splitter_pre.get_trajectory_groups()
    _selected_trajs  = sorted(_traj_groups_pre)[:MAX_CLIENTS]
    _load_indices    = set(i for t in _selected_trajs for i in _traj_groups_pre[t])
    print(f'MAX_CLIENTS={MAX_CLIENTS}: loading {len(_load_indices)}/{len(ds)} samples '
          f'({len(_selected_trajs)} trajectories)')
else:
    _load_indices = set(range(len(ds)))

# ── Fast preload: open each ZIP once, cache everything in RAM ────────────────
print('\nPreloading samples into RAM...')
_t0 = time.time()
_preloaded: Dict[int, tuple] = {}
_rng = np.random.default_rng(42)

# ── Phase 1: batch-load channel data — open each channel ZIP once ────────────
_channel_zip_groups = defaultdict(list)
for _i in range(len(ds)):
    if _i not in _load_indices:
        continue
    ref = ds.index[_i]
    _channel_zip_groups[str(ref.zip_path)].append(_i)

_channel_cache: Dict[int, tuple] = {}   # {idx: (csi, labels)}
for _zp, _idxs in _channel_zip_groups.items():
    with zipfile.ZipFile(_zp, 'r') as _z:
        for _i in _idxs:
            ref = ds.index[_i]
            raw = _z.read(ref.inner_npz)
            npz = np.load(io.BytesIO(raw))
            arrays = {k: npz[k] for k in npz.files}
            a_sq = np.squeeze(arrays['a']).astype(np.complex64)
            if a_sq.ndim == 2:
                a_sq = a_sq[:, :, np.newaxis]
            csi = ds._extract_csi(arrays)
            beam_idx, g_opt, H = ds.compute_beam_index(a_sq)
            # Extract los (line-of-sight flag) from raw npz if present
            _los_raw = arrays.get('los', None)
            los_val  = int(np.squeeze(_los_raw).flat[0]) if _los_raw is not None else 0
            _channel_cache[_i] = (csi, {
                'beam_index': np.array(beam_idx, dtype=np.int64),
                'g_opt'     : float(g_opt),
                'los'       : los_val,
                'H_complex' : H,
            })
    print(f'  Channel ZIP done: {Path(_zp).name}')
print(f'Channel cache: {len(_channel_cache)} samples')

# ── Phase 2: group sensor samples by ZIP ─────────────────────────────────────
_zip_groups = defaultdict(list)
_no_sensor  = []
for _i in range(len(mm_ds)):
    if _i not in _load_indices:
        continue
    resolved = mm_ds._sensor_map[_i]
    if resolved is not None:
        entry, cav_id, frame_id = resolved
        _zip_groups[str(entry.zip_path)].append((_i, entry, cav_id, frame_id))
    else:
        _no_sensor.append(_i)

# Samples without sensor data — use cached channel, zero sensor arrays
for _i in _no_sensor:
    csi, labels = _channel_cache[_i]
    _preloaded[_i] = (
        csi,
        np.zeros((_N_LIDAR_PTS, 3), np.float32),
        np.zeros(_IMU_DIM, np.float32),
        labels,
    )

# ── Phase 3: batch-load sensor ZIPs (channel already cached) ─────────────────
_done = len(_no_sensor)
for _zp, _samples in _zip_groups.items():
    with zipfile.ZipFile(_zp, 'r') as _z:
        for _i, _entry, _cav_id, _frame_id in _samples:
            csi, labels = _channel_cache[_i]
            prefix = f'{_entry.scenario_prefix}/{_cav_id}/{_frame_id}'
            try:
                pts = _read_pcd_xyz(_z.read(f'{prefix}.pcd'))
                if len(pts) == 0: pts = np.zeros((1, 3), np.float32)
                if len(pts) >= _N_LIDAR_PTS:
                    idx_pts = _rng.choice(len(pts), _N_LIDAR_PTS, replace=False)
                    pts = pts[idx_pts]
                else:
                    pts = np.concatenate([pts, np.zeros((_N_LIDAR_PTS - len(pts), 3), np.float32)])
                mean = pts.mean(0, keepdims=True); std = pts.std(0, keepdims=True) + 1e-8
                lidar = ((pts - mean) / std).astype(np.float32)
                imu   = _fast_parse_imu(_z.read(f'{prefix}.yaml'))
            except Exception:
                lidar = np.zeros((_N_LIDAR_PTS, 3), np.float32)
                imu   = np.zeros(_IMU_DIM, np.float32)
            _preloaded[_i] = (csi, lidar, imu, labels)
            _done += 1
            if _done % 500 == 0: print(f'  {_done}/{len(_load_indices)}')
    print(f'  Sensor ZIP done: {Path(_zp).name}')

del _channel_cache  # free intermediate cache
print(f'Preloaded {len(_preloaded)}/{len(ds)} samples in {time.time()-_t0:.1f}s')

  [OK] sunny -> /Users/sadmanrahin/Library/CloudStorage/GoogleDrive-saddyrahin2004@gmail.com/My Drive/Dataset/sunny/Channel Data/V2I/Nt_1_16_Nr_1_16_fc_28GHz
  [OK] foggy -> /Users/sadmanrahin/Library/CloudStorage/GoogleDrive-saddyrahin2004@gmail.com/My Drive/Dataset/foggy/Channel Data/Nt_1_16_Nr_1_16_fc_28GHz
  [OK] rainy -> /Users/sadmanrahin/Library/CloudStorage/GoogleDrive-saddyrahin2004@gmail.com/My Drive/Dataset/rainy/Channel Data/V2I/Nt_1_16_Nr_1_16_fc_28GHz
Channel dataset: 12330 samples
[SensorIndex] 48 sensor zips indexed
[MultimodalDataset] 12330 samples | sensor-aligned: 11730 | zeros: 600
Metadata index: 12330 samples
MAX_CLIENTS=50: loading 5350/12330 samples (50 trajectories)

Preloading samples into RAM...
  Channel ZIP done: Town03.zip
  Channel ZIP done: Town05.zip
  Channel ZIP done: Town10.zip
  Channel ZIP done: Town03.zip
Channel cache: 5350 samples
  500/5350
  Sensor ZIP done: Town03_5wayroad_seed28.zip
  1000/5350
  Sensor ZIP done: Town03_Tjunction_wiz_slope_s

In [22]:
# ── Build Windowed Clients ───────────────────────────────────────────────────

W = CFG['window_size']

windowed_clients = build_windowed_clients(
    preloaded_cache      = _preloaded,
    channel_dataset      = ds,
    window_size          = W,
    train_ratio          = 0.7,
    min_trajectory_length = 20,
)

if MAX_CLIENTS is not None:
    windowed_clients = windowed_clients[:MAX_CLIENTS]
    print(f'Capped to {len(windowed_clients)} windowed clients (MAX_CLIENTS={MAX_CLIENTS})')

print(f'\nWindow size W = {W}')
print('\n' + '=' * 70)
print(f'{"Clt":>4} | {"Trajectory ID":<48} | {"Train W":>7} | {"Test W":>6}')
print('-' * 70)
for c in windowed_clients[:20]:
    print(f'{c.client_id:>4} | {c.trajectory_id:<48} | '
          f'{len(c.train_windows):>7} | {len(c.test_windows):>6}')
if len(windowed_clients) > 20:
    print(f'  ... ({len(windowed_clients) - 20} more clients)')
print('=' * 70)

Metadata index: 12330 samples
Windowed clients: 50  (skipped: 66)
Total train windows: 3545  |  test windows: 1405
Avg windows/client:  99
Capped to 50 windowed clients (MAX_CLIENTS=50)

Window size W = 5

 Clt | Trajectory ID                                    | Train W | Test W
----------------------------------------------------------------------
   0 | foggy_Town03_Town03_5wayroad_cav_1               |      73 |     29
   1 | foggy_Town03_Town03_5wayroad_cav_2               |      73 |     29
   2 | foggy_Town03_Town03_5wayroad_cav_3               |      73 |     29
   3 | foggy_Town03_Town03_Tjunction_cav_1              |      66 |     26
   4 | foggy_Town03_Town03_Tjunction_cav_2              |      66 |     26
   5 | foggy_Town03_Town03_Tjunction_cav_3              |      66 |     26
   6 | foggy_Town03_Town03_Tjunction_cav_4              |      66 |     26
   7 | foggy_Town03_Town03_crossroad_cav_1              |      66 |     26
   8 | foggy_Town03_Town03_crossroad_cav_2      

In [23]:
# ── Federated Training — Spatio-Temporal (Batch-Window) ──────────────────────
# All window arrays are precomputed ONCE here, before start_simulation.
# No stacking, no ZIP I/O, no caching logic inside the client.

nr          = ds.nr
nt          = ds.nt
n_beams     = CFG['Q_tx']
window_size = CFG['window_size']

if ray.is_initialized():
    ray.shutdown(); print('Ray shutdown')

ray.init(num_cpus=4, num_gpus=0, include_dashboard=False, ignore_reinit_error=True)

# ── Precompute all client arrays ONCE (no per-round stacking) ─────────────────
print(f'Precomputing window arrays for {len(windowed_clients)} clients...')
_t_pre = time.time()
_client_data = []
for i, c in enumerate(windowed_clients):
    train_d = _load_windows_from_cache(_preloaded, c.train_windows)
    test_d  = _load_windows_from_cache(_preloaded, c.test_windows, return_H=True)
    _client_data.append({
        'train_data'   : train_d,
        'test_data'    : test_d,
        'tx_codebook'  : c.tx_codebook,
        'rx_codebook'  : c.rx_codebook,
        'trajectory_id': c.trajectory_id,
    })
    if (i + 1) % 10 == 0 or (i + 1) == len(windowed_clients):
        print(f'  {i+1}/{len(windowed_clients)}')
print(f'Precomputed in {time.time()-_t_pre:.1f}s  '
      f'— {sum(d["train_data"][0][0].nbytes + d["test_data"][0][0].nbytes for d in _client_data)/1e6:.0f} MB')
print(f'  {len(_client_data)} clients ready (direct closure access — no nested ray.get)')


def st_client_fn(context: fl.common.Context) -> fl.client.Client:
    tf.config.threading.set_intra_op_parallelism_threads(1)
    tf.config.threading.set_inter_op_parallelism_threads(1)

    # Direct list access — avoids nested ray.get() inside a Flower actor,
    # which causes "Invalid type of object refs, NoneType" crashes.
    client_idx = int(context.node_id) % len(_client_data)
    data = _client_data[client_idx]

    model = SpatioTemporalBeamModel(
        nr=nr, nt=nt, n_beams=n_beams, window_size=window_size,
        emb_dim=CFG['emb_dim'], gru_units=CFG['gru_units'], dropout=CFG['dropout'],
    )
    model.build_model(nr, nt, window_size)

    return SpatioTemporalFlowerClient(
        model          = model,
        train_data     = data['train_data'],
        test_data      = data['test_data'],
        tx_codebook    = data['tx_codebook'],
        rx_codebook    = data['rx_codebook'],
        cfg            = CFG,
        trajectory_id  = data['trajectory_id'],
    ).to_client()


tf.keras.backend.clear_session()
_global_model = SpatioTemporalBeamModel(
    nr=nr, nt=nt, n_beams=n_beams, window_size=window_size,
    emb_dim=CFG['emb_dim'], gru_units=CFG['gru_units'], dropout=CFG['dropout'],
)
_global_model.build_model(nr, nt, window_size)


def agg_metrics(metrics):
    agg = {}
    for _, m in metrics:
        for k, v in m.items():
            agg.setdefault(k, []).append(v)
    return {k: float(np.mean(v)) for k, v in agg.items()
            if v and isinstance(v[0], (int, float, np.integer, np.floating))}


st_strategy = fl.server.strategy.FedAvg(
    fraction_fit                    = CFG['client_frac'],
    fraction_evaluate               = CFG['client_frac'],
    min_fit_clients                 = len(_client_data),
    min_evaluate_clients            = len(_client_data),
    min_available_clients           = len(_client_data),
    initial_parameters              = fl.common.ndarrays_to_parameters(
                                          _global_model.get_weights()),
    fit_metrics_aggregation_fn      = agg_metrics,
    evaluate_metrics_aggregation_fn = agg_metrics,
)

ST_ROUNDS = 40
print(f'\nStarting batch-window FL: {ST_ROUNDS} rounds, {len(_client_data)} clients')

st_history = fl.simulation.start_simulation(
    client_fn        = st_client_fn,
    num_clients      = len(_client_data),
    config           = fl.server.ServerConfig(num_rounds=ST_ROUNDS),
    strategy         = st_strategy,
    client_resources = {'num_cpus': 2, 'num_gpus': 0.0},
)
print('Batch-window FL complete.')

Ray shutdown


2026-03-05 04:25:48,642	INFO worker.py:1771 -- Started a local Ray instance.


Precomputing window arrays for 50 clients...
  10/50
  20/50
  30/50
  40/50
  50/50
Precomputed in 0.4s  — 76 MB
  50 clients ready (direct closure access — no nested ray.get)


	Instead, use the `flwr run` CLI command to start a local simulation in your Flower app, as shown for example below:

		$ flwr new  # Create a new Flower app from a template

		$ flwr run  # Run the Flower app in Simulation Mode

	Using `start_simulation()` is deprecated.

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        
INFO :      Starting Flower simulation, config: num_rounds=40, no round_timeout


SpatioTemporalBeamModel built — 1,351,281 parameters

Starting batch-window FL: 40 rounds, 50 clients


...

(raylet) [2026-03-05 05:41:41,556 E 14885 2751734] (raylet) file_system_monitor.cc:111: /tmp/ray/session_2026-03-05_04-25-52_801228_84604 is over 95% full, available space: 15896334336; capacity: 494384795648. Object creation will fail if spilling is required.


(ClientAppActor pid=14905) SpatioTemporalBeamModel built — 1,351,281 parameters


(raylet) [2026-03-05 05:41:51,567 E 14885 2751734] (raylet) file_system_monitor.cc:111: /tmp/ray/session_2026-03-05_04-25-52_801228_84604 is over 95% full, available space: 14820782080; capacity: 494384795648. Object creation will fail if spilling is required.


(ClientAppActor pid=14907) SpatioTemporalBeamModel built — 1,351,281 parameters
(ClientAppActor pid=14906) SpatioTemporalBeamModel built — 1,351,281 parameters


(raylet) [2026-03-05 05:42:01,657 E 14885 2751734] (raylet) file_system_monitor.cc:111: /tmp/ray/session_2026-03-05_04-25-52_801228_84604 is over 95% full, available space: 14799536128; capacity: 494384795648. Object creation will fail if spilling is required.
(raylet) [2026-03-05 05:42:11,742 E 14885 2751734] (raylet) file_system_monitor.cc:111: /tmp/ray/session_2026-03-05_04-25-52_801228_84604 is over 95% full, available space: 14797148160; capacity: 494384795648. Object creation will fail if spilling is required.
(raylet) [2026-03-05 05:42:21,825 E 14885 2751734] (raylet) file_system_monitor.cc:111: /tmp/ray/session_2026-03-05_04-25-52_801228_84604 is over 95% full, available space: 14796980224; capacity: 494384795648. Object creation will fail if spilling is required.
(raylet) [2026-03-05 05:42:31,886 E 14885 2751734] (raylet) file_system_monitor.cc:111: /tmp/ray/session_2026-03-05_04-25-52_801228_84604 is over 95% full, available space: 15874781184; capacity: 494384795648. Object 

In [ ]:
# ── Results Visualization ─────────────────────────────────────────────────────
import matplotlib
matplotlib.use('Agg')  # headless for server environments
import matplotlib.pyplot as plt

metrics_eval = st_history.metrics_distributed

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle(f'Spatio-Temporal FL  (W={CFG["window_size"]}, GRU={CFG["gru_units"]}, '
             f'{len(_client_data)} clients)', fontsize=13)

plot_keys = [
    ('beam_accuracy',        'Beam Top-1 Accuracy',   'steelblue'),
    ('beam_top3_accuracy',   'Beam Top-3 Accuracy',   'teal'),
    ('normalized_gain',      'Normalised Beam Gain',  'darkorange'),
    ('beam_flicker_rate',    'Beam Flicker Rate',      'crimson'),
    ('beam_num_unique_preds','Unique Beam Predictions','purple'),
    ('loss',                 'Cross-Entropy Loss',    'grey'),
]

for ax, (key, title, color) in zip(axes.flat, plot_keys):
    if key in metrics_eval:
        rounds = [r for r, _ in metrics_eval[key]]
        vals   = [v for _, v in metrics_eval[key]]
        ax.plot(rounds, vals, color=color, linewidth=2)
        ax.set_title(title); ax.set_xlabel('Round'); ax.grid(alpha=0.3)
    else:
        ax.set_title(f'{title} (no data)'); ax.axis('off')

plt.tight_layout()
plt.savefig('st_fl_results.png', dpi=120)
plt.show()
print('Plot saved to st_fl_results.png')

# ── Final round summary ─────────────────────────────────────────────────────
print('\n=== Final round metrics ===')
for key, title, _ in plot_keys:
    if key in metrics_eval and metrics_eval[key]:
        _, val = metrics_eval[key][-1]
        print(f'  {title:<30} {val:.4f}')


---
## Buffered Stream — Slide-and-Train

Each UAV client maintains a **circular buffer** of the last `buffer_size` frames.
Every FL round, `chunk_size` new frames are "revealed" (simulating data collected
since the last sync). The client trains only on windows from the current buffer.

**Why this matters for DDIL deployments:**
- **Memory**: UAV onboard RAM is bounded — the buffer enforces that limit.
- **Relevance**: Old frames (different altitude / heading) shouldn't dominate.
- **Communication**: Local epoch time is short and *predictable* (fixed buffer ≈ fixed window count).

**Stateless clients**: The buffer for round `r` is always `train_indices[max(0, r·chunk - buffer) : r·chunk]`.
No persistent state is needed — clients can be torn down and rebuilt between rounds.


In [14]:
# ── Streaming Config ─────────────────────────────────────────────────────────
# Inherits all base CFG values; adds buffer / chunk parameters.
# Client count is controlled by MAX_CLIENTS in cell 3 (the CFG cell).

STREAMING_CFG = {
    **CFG,
    'buffer_size' : 50,
    'chunk_size'  : 10,
}

print(f'buffer_size          : {STREAMING_CFG["buffer_size"]} frames')
print(f'chunk_size           : {STREAMING_CFG["chunk_size"]} frames / round')
print(f'max windows          : {STREAMING_CFG["buffer_size"] - CFG["window_size"] + 1} (= buffer_size - W + 1)')
print(f'rounds to fill buffer: {STREAMING_CFG["buffer_size"] // STREAMING_CFG["chunk_size"]} (= buffer_size / chunk_size)')
print(f'MAX_CLIENTS          : {MAX_CLIENTS}  (set in cell 3)')

buffer_size          : 50 frames
chunk_size           : 10 frames / round
max windows          : 46 (= buffer_size - W + 1)
rounds to fill buffer: 5 (= buffer_size / chunk_size)
MAX_CLIENTS          : 3  (set in cell 3)


In [15]:
# ── StreamingClientData + build_streaming_clients ─────────────────────────────


@dataclass
class StreamingClientData:
    """Client data for streaming FL — holds raw sorted frame indices, not pre-built windows."""
    train_indices_sorted: List[int]   # frames sorted by frame_id; client chunks these
    test_windows:         List[List[int]]  # pre-built test windows (eval doesn't change)
    tx_codebook:          np.ndarray
    rx_codebook:          np.ndarray
    client_id:            int
    trajectory_id:        str


def build_streaming_clients(
    preloaded_cache: dict,
    channel_dataset: ChannelDataset,
    window_size: int = 5,
    train_ratio: float = 0.7,
    min_trajectory_length: int = 20,
) -> List[StreamingClientData]:
    """
    Build streaming FL clients.

    Unlike build_windowed_clients, the train set is returned as a sorted list
    of frame indices — the client decides which frames to use each round
    based on its circular buffer logic.

    Test windows are still pre-built (evaluation is always over the full test
    trajectory to give a consistent, comparable metric each round).
    """
    splitter    = DatasetSplitter(channel_dataset)
    traj_groups = splitter.get_trajectory_groups()
    tx_cb       = channel_dataset.tx_codebook
    rx_cb       = channel_dataset.rx_codebook

    clients, skipped = [], 0
    min_len = max(min_trajectory_length, window_size * 2 + 2)

    for traj_id in sorted(traj_groups):
        indices = [i for i in traj_groups[traj_id] if i in preloaded_cache]
        if len(indices) < min_len:
            skipped += 1; continue

        # Temporal split — test frames are always the last 30%
        train_idx, test_idx = splitter.split_trajectory_temporal(indices, train_ratio)

        if len(train_idx) < window_size or len(test_idx) < window_size:
            skipped += 1; continue

        test_windows = [test_idx[s:s + window_size]
                        for s in range(len(test_idx) - window_size + 1)]
        if not test_windows:
            skipped += 1; continue

        clients.append(StreamingClientData(
            train_indices_sorted = train_idx,  # already sorted by frame_id
            test_windows         = test_windows,
            tx_codebook          = tx_cb,
            rx_codebook          = rx_cb,
            client_id            = len(clients),
            trajectory_id        = traj_id,
        ))

    total_train = sum(len(c.train_indices_sorted) for c in clients)
    total_test  = sum(len(c.test_windows)         for c in clients)
    print(f'Streaming clients: {len(clients)}  (skipped: {skipped})')
    print(f'Total train frames: {total_train}  |  total test windows: {total_test}')
    return clients


print('_load_windows_from_cache, StreamingClientData, build_streaming_clients defined.')


_load_windows_from_cache, StreamingClientData, build_streaming_clients defined.


In [16]:
class StreamingSTFlowerClient(fl.client.NumPyClient):
    """
    Streaming Spatio-Temporal FL client.

    fit()     : rebuilds train windows each round (buffer contents change — intentional)
    evaluate(): uses pre-stacked test_data arrays (never changes — no rebuild needed)

    Stateless: buffer is deterministically reconstructed from current_round.
    GRU hidden state resets each mini-batch (stateful=False, Keras default).
    """

    def __init__(self, model, preloaded_cache, train_indices_sorted,
                 test_data, tx_codebook, rx_codebook, cfg, trajectory_id):
        """
        Parameters
        ----------
        train_indices_sorted : List[int] sorted by frame_id (client chunks per round)
        test_data            : ((csi, lidar, imu), y, H) — pre-stacked, never rebuilt
        """
        self.model         = model
        self.cache         = preloaded_cache
        self.train_indices = train_indices_sorted
        self.test_data     = test_data          # pre-stacked, reused every evaluate()
        self.tx_codebook   = tx_codebook
        self.rx_codebook   = rx_codebook
        self.cfg           = cfg
        self.trajectory_id = trajectory_id
        self.optimizer     = keras.optimizers.Adam(cfg['lr'])
        self.grad_clip_norm = cfg.get('grad_clip_norm', 5.0)
        self.batch_size    = cfg.get('batch_size', 32)
        self.buffer_size   = cfg.get('buffer_size', 50)
        self.chunk_size    = cfg.get('chunk_size', 10)
        self.window_size   = model.window_size

    # ── Buffer logic ──────────────────────────────────────────────────────
    def _get_buffer_windows(self, round_num: int) -> List[List[int]]:
        """
        Circular buffer at round `round_num`:
            revealed = train_indices[:round_num * chunk_size]
            buffer   = last buffer_size frames of revealed
        Once all frames are revealed, stays on the last buffer_size frames.
        """
        n         = len(self.train_indices)
        end_idx   = min(round_num * self.chunk_size, n)
        start_idx = max(0, end_idx - self.buffer_size)
        buffer    = self.train_indices[start_idx:end_idx]
        if len(buffer) < self.window_size:
            return []
        return [buffer[s:s + self.window_size]
                for s in range(len(buffer) - self.window_size + 1)]

    def _ensure_built(self):
        if getattr(self.model, 'built', False): return
        self.model.build_model(self.model.nr, self.model.nt, self.window_size)

    def compute_normalized_gain(self, H_batch, pred_indices):
        gains = []
        for i in range(len(pred_indices)):
            resp = self.rx_codebook.conj().T @ H_batch[i] @ self.tx_codebook
            g    = np.abs(resp) ** 2
            gains.append(g[:, pred_indices[i]].max() / (g.max() + 1e-12))
        return float(np.mean(gains))

    def get_parameters(self, config):
        self._ensure_built()
        return self.model.get_weights()

    def fit(self, parameters, config):
        self._ensure_built()
        self.model.set_weights(parameters)

        round_num = int(config.get('current_round', 1))
        windows   = self._get_buffer_windows(round_num)

        if not windows:
            return self.model.get_weights(), 0, {
                'loss': 0.0, 'buffer_windows': 0,
                'trajectory_id': self.trajectory_id,
            }

        # Buffer changes each round — must rebuild (pure RAM numpy ops, ~ms)
        (csi_arr, lidar_arr, imu_arr), y_train = _load_windows_from_cache(
            self.cache, windows
        )

        n, bs = len(csi_arr), self.batch_size
        ep_losses = []

        for _ in range(self.cfg['local_epochs']):
            perm = np.random.permutation(n)
            batch_losses = []
            for start in range(0, n, bs):
                end     = min(start + bs, n)
                idx     = perm[start:end]
                csi_b   = tf.constant(csi_arr[idx],   tf.float32)
                lidar_b = tf.constant(lidar_arr[idx], tf.float32)
                imu_b   = tf.constant(imu_arr[idx],   tf.float32)
                y_b     = tf.constant(y_train['beam_index'][idx], tf.int32)

                with tf.GradientTape() as tape:
                    logits = self.model(csi_b, lidar_b, imu_b, training=True)
                    loss   = tf.reduce_mean(
                        tf.nn.sparse_softmax_cross_entropy_with_logits(
                            labels=y_b, logits=logits))

                grads   = tape.gradient(loss, self.model.trainable_weights)
                safe    = [g if g is not None else tf.zeros_like(v)
                           for g, v in zip(grads, self.model.trainable_weights)]
                clipped, _ = tf.clip_by_global_norm(safe, self.grad_clip_norm)
                self.optimizer.apply_gradients(
                    zip(clipped, self.model.trainable_weights))
                batch_losses.append(float(loss.numpy()))
            ep_losses.append(np.mean(batch_losses))

        return self.model.get_weights(), len(windows), {
            'loss'          : ep_losses[-1],
            'buffer_windows': len(windows),
            'trajectory_id' : self.trajectory_id,
        }

    def evaluate(self, parameters, config):
        self._ensure_built()
        self.model.set_weights(parameters)

        # test_data is pre-stacked — no rebuild, no stacking, no I/O
        (csi_arr, lidar_arr, imu_arr), y_test, H_test = self.test_data
        bs, all_logits = self.batch_size, []

        for start in range(0, len(csi_arr), bs):
            end = min(start + bs, len(csi_arr))
            logits = self.model(
                tf.constant(csi_arr[start:end],   tf.float32),
                tf.constant(lidar_arr[start:end], tf.float32),
                tf.constant(imu_arr[start:end],   tf.float32),
                training=False,
            )
            all_logits.append(logits)

        logits   = tf.concat(all_logits, axis=0)
        y_tf     = tf.constant(y_test['beam_index'], dtype=tf.int32)
        loss     = tf.reduce_mean(
            tf.nn.sparse_softmax_cross_entropy_with_logits(labels=y_tf, logits=logits))
        preds    = tf.argmax(logits, 1).numpy()
        acc      = float(np.mean(preds == y_test['beam_index']))
        top3     = tf.math.in_top_k(targets=y_tf, predictions=logits, k=3)
        top3_acc = float(tf.reduce_mean(tf.cast(top3, tf.float32)).numpy())
        eval_n   = min(len(preds), 200)
        norm_gain = self.compute_normalized_gain(H_test[:eval_n], preds[:eval_n])
        flicker  = float(np.mean(preds[1:] != preds[:-1])) if len(preds) > 1 else 0.0

        return float(loss.numpy()), len(csi_arr), {
            'beam_accuracy'        : acc,
            'beam_top3_accuracy'   : top3_acc,
            'normalized_gain'      : norm_gain,
            'beam_flicker_rate'    : flicker,
            'beam_num_unique_preds': int(len(np.unique(preds))),
            'loss'                 : float(loss.numpy()),
        }


print('StreamingSTFlowerClient defined.')
print('  fit()     : buffer windows rebuilt per round (RAM-only numpy ops)')
print('  evaluate(): uses pre-stacked test_data — zero rebuild cost')


StreamingSTFlowerClient defined.
  fit()     : buffer windows rebuilt per round (RAM-only numpy ops)
  evaluate(): uses pre-stacked test_data — zero rebuild cost


In [17]:
# ── Build Streaming Clients ───────────────────────────────────────────────────
# Reuses _preloaded cache and channel dataset from the dataset loading cell.

streaming_clients = build_streaming_clients(
    preloaded_cache      = _preloaded,
    channel_dataset      = ds,
    window_size          = CFG['window_size'],
    train_ratio          = 0.7,
    min_trajectory_length= 20,
)

if MAX_CLIENTS is not None:
    streaming_clients = streaming_clients[:MAX_CLIENTS]
    print(f'  Capped to {len(streaming_clients)} streaming clients (MAX_CLIENTS={MAX_CLIENTS})')

# Quick sanity check
c0 = streaming_clients[0]
print(f'\nClient 0 — {c0.trajectory_id}')
print(f'  Train frames:  {len(c0.train_indices_sorted)}')
print(f'  Test windows:  {len(c0.test_windows)}')

chunk     = STREAMING_CFG['chunk_size']
buf       = STREAMING_CFG['buffer_size']
W         = CFG['window_size']
print(f'\nBuffer simulation (W={W}, chunk={chunk}, buf={buf}):')
for r in [1, 2, 5, buf//chunk, buf//chunk+1]:
    idx = c0.train_indices_sorted
    end = min(r * chunk, len(idx))
    sta = max(0, end - buf)
    n_win = max(0, (end - sta) - W + 1)
    print(f'  Round {r:>3}: buffer frames [{sta:>4}:{end:>4}]  → {n_win} windows')

Metadata index: 12330 samples
Streaming clients: 3  (skipped: 113)
Total train frames: 231  |  total test windows: 87
  Capped to 3 streaming clients (MAX_CLIENTS=3)

Client 0 — foggy_Town03_Town03_5wayroad_cav_1
  Train frames:  77
  Test windows:  29

Buffer simulation (W=5, chunk=10, buf=50):
  Round   1: buffer frames [   0:  10]  → 6 windows
  Round   2: buffer frames [   0:  20]  → 16 windows
  Round   5: buffer frames [   0:  50]  → 46 windows
  Round   5: buffer frames [   0:  50]  → 46 windows
  Round   6: buffer frames [  10:  60]  → 46 windows


In [ ]:
# ── Federated Training — Streaming (Buffered Slide-and-Train) ────────────────
# Test arrays precomputed ONCE. Train windows rebuilt per round (buffer changes).
# _preloaded is accessed directly from the closure — no ray.put/ray.get — which
# avoids "owner unknown" crashes when Ray sessions are restarted between cells.

nr          = ds.nr
nt          = ds.nt
n_beams     = CFG['Q_tx']
window_size = CFG['window_size']

if ray.is_initialized():
    ray.shutdown(); print('Ray shutdown')

ray.init(num_cpus=4, num_gpus=0, include_dashboard=False, ignore_reinit_error=True)

# ── Precompute test arrays ONCE per streaming client ─────────────────────────
print(f'Precomputing test arrays for {len(streaming_clients)} streaming clients...')
_t_pre = time.time()
_stream_client_data = []
for i, c in enumerate(streaming_clients):
    test_d = _load_windows_from_cache(_preloaded, c.test_windows, return_H=True)
    _stream_client_data.append({
        'train_indices_sorted': c.train_indices_sorted,  # indices only — rebuilt per round
        'test_data'           : test_d,                  # arrays — never rebuilt
        'tx_codebook'         : c.tx_codebook,
        'rx_codebook'         : c.rx_codebook,
        'trajectory_id'       : c.trajectory_id,
    })
    if (i + 1) % 10 == 0 or (i + 1) == len(streaming_clients):
        print(f'  {i+1}/{len(streaming_clients)}')
print(f'Test arrays precomputed in {time.time()-_t_pre:.1f}s')
print(f'  {len(_stream_client_data)} streaming clients  |  '
      f'{len(_preloaded)} samples in closure (no ray.put)')


def stream_client_fn(context: fl.common.Context) -> fl.client.Client:
    tf.config.threading.set_intra_op_parallelism_threads(1)
    tf.config.threading.set_inter_op_parallelism_threads(1)

    # Direct list access — no nested ray.get(), no stale ObjectRef.
    client_idx = int(context.node_id) % len(_stream_client_data)
    data = _stream_client_data[client_idx]

    # _preloaded is a plain dict of numpy arrays — safe to capture in closure.
    # Flower workers share the parent process memory via fork (COW), so this
    # does not copy the full cache into every actor.
    model = SpatioTemporalBeamModel(
        nr=nr, nt=nt, n_beams=n_beams, window_size=window_size,
        emb_dim=CFG['emb_dim'], gru_units=CFG['gru_units'], dropout=CFG['dropout'],
    )
    model.build_model(nr, nt, window_size)

    return StreamingSTFlowerClient(
        model                = model,
        preloaded_cache      = _preloaded,            # direct dict — no ObjectRef
        train_indices_sorted = data['train_indices_sorted'],
        test_data            = data['test_data'],
        tx_codebook          = data['tx_codebook'],
        rx_codebook          = data['rx_codebook'],
        cfg                  = STREAMING_CFG,
        trajectory_id        = data['trajectory_id'],
    ).to_client()


tf.keras.backend.clear_session()
_stream_global_model = SpatioTemporalBeamModel(
    nr=nr, nt=nt, n_beams=n_beams, window_size=window_size,
    emb_dim=CFG['emb_dim'], gru_units=CFG['gru_units'], dropout=CFG['dropout'],
)
_stream_global_model.build_model(nr, nt, window_size)


def agg_metrics_stream(metrics):
    agg = {}
    for _, m in metrics:
        for k, v in m.items():
            agg.setdefault(k, []).append(v)
    return {k: float(np.mean(v)) for k, v in agg.items()
            if v and isinstance(v[0], (int, float, np.integer, np.floating))}


stream_strategy = fl.server.strategy.FedAvg(
    fraction_fit                    = STREAMING_CFG['client_frac'],
    fraction_evaluate               = STREAMING_CFG['client_frac'],
    min_fit_clients                 = len(_stream_client_data),
    min_evaluate_clients            = len(_stream_client_data),
    min_available_clients           = len(_stream_client_data),
    initial_parameters              = fl.common.ndarrays_to_parameters(
                                          _stream_global_model.get_weights()),
    on_fit_config_fn                = lambda server_round: {'current_round': server_round},
    fit_metrics_aggregation_fn      = agg_metrics_stream,
    evaluate_metrics_aggregation_fn = agg_metrics_stream,
)

STREAM_ROUNDS = 10
print(f'\nStarting Streaming FL: {STREAM_ROUNDS} rounds, {len(_stream_client_data)} clients')
print(f'buffer={STREAMING_CFG["buffer_size"]} frames, '
      f'chunk={STREAMING_CFG["chunk_size"]} frames/round, '
      f'W={window_size}')

stream_history = fl.simulation.start_simulation(
    client_fn        = stream_client_fn,
    num_clients      = len(_stream_client_data),
    config           = fl.server.ServerConfig(num_rounds=STREAM_ROUNDS),
    strategy         = stream_strategy,
    client_resources = {'num_cpus': 4, 'num_gpus': 0.0},
)
print('Streaming FL complete.')

Ray shutdown


2026-03-05 02:09:31,746	INFO worker.py:1771 -- Started a local Ray instance.


Precomputing test arrays for 3 streaming clients...
  3/3
Test arrays precomputed in 0.0s
  3 streaming clients  |  330 samples in closure (no ray.put)


	Instead, use the `flwr run` CLI command to start a local simulation in your Flower app, as shown for example below:

		$ flwr new  # Create a new Flower app from a template

		$ flwr run  # Run the Flower app in Simulation Mode

	Using `start_simulation()` is deprecated.

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        
INFO :      Starting Flower simulation, config: num_rounds=10, no round_timeout


SpatioTemporalBeamModel built — 1,351,281 parameters

Starting Streaming FL: 10 rounds, 3 clients
buffer=50 frames, chunk=10 frames/round, W=5


...

INFO :      aggregate_evaluate: received 3 results and 0 failures
INFO :      
INFO :      [ROUND 4]
INFO :      configure_fit: strategy sampled 3 clients (out of 3)


(ClientAppActor pid=84290) SpatioTemporalBeamModel built — 1,351,281 parameters [repeated 4x across cluster]


INFO :      aggregate_fit: received 3 results and 0 failures
INFO :      configure_evaluate: strategy sampled 3 clients (out of 3)


(ClientAppActor pid=84290) SpatioTemporalBeamModel built — 1,351,281 parameters
(ClientAppActor pid=84289) SpatioTemporalBeamModel built — 1,351,281 parameters


INFO :      aggregate_evaluate: received 3 results and 0 failures
INFO :      
INFO :      [ROUND 5]
INFO :      configure_fit: strategy sampled 3 clients (out of 3)


In [ ]:
# ── Comparison: Batch-Window vs Streaming ─────────────────────────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Batch-Window (blue) vs Streaming Circular Buffer (orange)', fontsize=13)

plot_keys = [
    ('beam_accuracy',        'Beam Top-1 Accuracy'),
    ('beam_top3_accuracy',   'Beam Top-3 Accuracy'),
    ('normalized_gain',      'Normalised Beam Gain'),
    ('beam_flicker_rate',    'Beam Flicker Rate  ↓ better'),
    ('beam_num_unique_preds','Unique Beam Predictions'),
    ('loss',                 'Cross-Entropy Loss'),
]

# Both histories use the same metric keys
histories = []
if 'st_history' in dir():     histories.append(('Batch-Window', st_history,     'steelblue'))
if 'stream_history' in dir(): histories.append(('Streaming',    stream_history, 'darkorange'))

for ax, (key, title) in zip(axes.flat, plot_keys):
    ax.set_title(title); ax.set_xlabel('Round'); ax.grid(alpha=0.3)
    for label, hist, color in histories:
        m = hist.metrics_distributed
        if key in m:
            rounds = [r for r, _ in m[key]]
            vals   = [v for _, v in m[key]]
            ax.plot(rounds, vals, label=label, color=color, linewidth=2)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('stream_vs_batch_comparison.png', dpi=120)
plt.show()
print('Saved: stream_vs_batch_comparison.png')

# ── Numeric summary ──────────────────────────────────────────────────────────
print('\n=== Final-round comparison ===')
print(f'{"Metric":<30} {"Batch-Window":>15} {"Streaming":>12}')
print('-' * 60)
for key, title in plot_keys:
    row = [f'{title:<30}']
    for label, hist, _ in histories:
        m = hist.metrics_distributed
        val = m[key][-1][1] if key in m and m[key] else float('nan')
        row.append(f'{val:>15.4f}')
    print(''.join(row))


In [ ]:
# Validation 1: g_opt sanity over random frames
_rng_val = np.random.default_rng(2026)
N = min(len(ds), 1000)
if N == 0:
    raise ValueError('Dataset is empty.')

sample_idx = _rng_val.choice(len(ds), size=N, replace=False)
g_opts = []

for _i in sample_idx:
    _, _labels = ds[int(_i)]
    g_opts.append(float(_labels['g_opt']))

g_opts = np.asarray(g_opts, dtype=np.float64)
print(f'g_opt stats over N={N}: min={g_opts.min():.6e}, mean={g_opts.mean():.6e}, max={g_opts.max():.6e}')
assert np.isfinite(g_opts).all() and (g_opts >= 0.0).all()


In [ ]:
# Validation 2: LOS sanity (tau + path-power based)
_rng_val2 = np.random.default_rng(2027)
N2 = min(len(ds), 1000)
if N2 == 0:
    raise ValueError('Dataset is empty.')

sample_idx2 = _rng_val2.choice(len(ds), size=N2, replace=False)
rows = []
for _i in sample_idx2:
    _, _labels = ds[int(_i)]
    _ref = ds.index[int(_i)]
    _arrays = ds._load_npz(_ref)
    _tau = np.asarray(_arrays['tau']).squeeze()
    if _tau.ndim == 0:
        _tau = np.array([float(_tau)])
    _l0 = int(np.argmin(_tau))
    rows.append({
        'sample_idx': int(_i),
        'argmin_tau_idx': _l0,
        'dom_ratio': float(_labels['dom_ratio']),
        'delay_spread': float(_labels['delay_spread']),
        'los': int(_labels['los']),
    })

los_df = pd.DataFrame(rows)
print(f'LOS fraction overall: {los_df["los"].mean():.4f}')
print('dom_ratio stats:',
      f"min={los_df['dom_ratio'].min():.4f}",
      f"mean={los_df['dom_ratio'].mean():.4f}",
      f"max={los_df['dom_ratio'].max():.4f}")
print('delay_spread stats (ns):',
      f"min={los_df['delay_spread'].min()*1e9:.3f}",
      f"mean={los_df['delay_spread'].mean()*1e9:.3f}",
      f"max={los_df['delay_spread'].max()*1e9:.3f}")

assert np.isfinite(los_df['dom_ratio']).all()
assert ((los_df['dom_ratio'] >= 0.0) & (los_df['dom_ratio'] <= 1.0 + 1e-6)).all()
assert np.isfinite(los_df['delay_spread']).all()
assert (los_df['delay_spread'] >= 0.0).all()

_meta = ds.build_metadata_index()
if {'sample_idx', 'town', 'location'}.issubset(_meta.columns):
    _merge = los_df.merge(_meta[['sample_idx', 'town', 'location']], on='sample_idx', how='left')
    print('\nLOS fraction by (town, location):')
    print(_merge.groupby(['town', 'location'])['los'].mean().sort_index().to_string())

print('\n10 random LOS examples:')
_show = _rng_val2.choice(len(los_df), size=min(10, len(los_df)), replace=False)
for _j in _show:
    _r = los_df.iloc[int(_j)]
    print(
        f"sample={int(_r['sample_idx'])} | argmin_tau_idx={int(_r['argmin_tau_idx'])} | "
        f"dom_ratio={_r['dom_ratio']:.3f} | delay_spread={_r['delay_spread']*1e9:.2f} ns | los={int(_r['los'])}"
    )


In [ ]:
# Validation 3: beam_change sanity on windowed data
_all_windows = [w for _c in windowed_clients for w in _c.train_windows]
if len(_all_windows) == 0:
    raise ValueError('No train windows available for beam_change validation.')

_rng_val3 = np.random.default_rng(2028)
max_windows = min(len(_all_windows), 5000)
_sel_idx = _rng_val3.choice(len(_all_windows), size=max_windows, replace=False)
_sel_windows = [_all_windows[int(i)] for i in _sel_idx]

_, _y_bc = _load_windows_from_cache(_preloaded, _sel_windows, normalize_csi=False)
beam_change_rate = float(np.mean(_y_bc['beam_change']))
print(f'beam_change rate over {max_windows} windows: {100.0 * beam_change_rate:.2f}%')

_traj = None
for _c in windowed_clients:
    if len(_c.train_windows) >= 30:
        _traj = _c
        break
if _traj is None:
    raise ValueError('Could not find a trajectory with >=30 windows.')

_traj_windows = _traj.train_windows[:30]
_, _y_traj = _load_windows_from_cache(_preloaded, _traj_windows, normalize_csi=False)
_beam_seq = [int(_preloaded[w[-1]][3]['beam_index']) for w in _traj_windows]
_beam_change_seq = _y_traj['beam_change'].astype(int).tolist()

print('beam_index sequence (~30):', _beam_seq)
print('beam_change sequence    :', _beam_change_seq)
print('beam_change positions   :', np.where(np.array(_beam_change_seq) == 1)[0].tolist())


In [ ]:
# Validation 4: train-time plumbing (X, y dict keys/shapes/dtypes)
_probe_windows = _all_windows[:max(1, min(len(_all_windows), 64))]
(X_probe, y_probe) = _load_windows_from_cache(_preloaded, _probe_windows, normalize_csi=False)

csi_arr, lidar_arr, imu_arr = X_probe
B = min(8, len(csi_arr))
X_batch = (csi_arr[:B], lidar_arr[:B], imu_arr[:B])
y_batch = {k: v[:B] for k, v in y_probe.items()}

print('X batch:')
print('  csi  :', X_batch[0].shape, X_batch[0].dtype)
print('  lidar:', X_batch[1].shape, X_batch[1].dtype)
print('  imu  :', X_batch[2].shape, X_batch[2].dtype)
print('y batch:')
for _k, _v in y_batch.items():
    print(f'  {_k}: {_v.shape} {_v.dtype}')

expected_output_names = ['beam_index', 'g_opt', 'los', 'beam_change']
print('y keys:', sorted(y_batch.keys()))
print('expected model output names:', sorted(expected_output_names))
assert set(y_batch.keys()) == set(expected_output_names)
